# 0.23 — Quantum with a **rolling (point-in-time) baseline**

Fixes the flaw in 0.22: a *fixed* 2019–20 baseline over a 3-year window goes stale — every term
first seen anywhere in 2021–23 stays "novel" forever → 47k novel entities, 1,312 promoted, an
835-node blob. Here "novel" is **point-in-time**: *not seen in the trailing 18 months at each week*.

**Emerge-then-track** (so tracking still works): an entity enters the candidate pool the week it is
rolling-novel (*emerges*), then is **tracked forward for a horizon** (so promotion can see its
growth) even after it is no longer novel. Old entities drop out after the horizon → no 3-year flood.

In [1]:
import os, re
from collections import Counter, defaultdict
from itertools import combinations
from pathlib import Path
import networkx as nx, numpy as np, pandas as pd
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUTPUT_DIR = _ROOT / "notebooks" / "output"
ENV_PATH = _ROOT / ".env"
if ENV_PATH.exists():
    for _l in ENV_PATH.read_text().splitlines():
        _l=_l.strip()
        if "=" in _l and not _l.startswith("#"):
            _k,_,_v=_l.partition("="); os.environ.setdefault(_k.strip(), _v.strip().strip('"').strip("'"))
DISCOVERY_START = pd.Timestamp("2021-01-01"); DISCOVERY_END = pd.Timestamp("2023-12-31")
WQTM = pd.Timestamp("2025-10-09")
FREQ="W-MON"; REP_HL=8
MIN_MENTIONS=3; DEGREE_MIN=8; PERSIST_WEEKS=2; CLUSTER_K=15; CLUSTER_MAX=0.60; LLM_MAX_GROUPS=25
ROLL_WEEKS=78        # trailing window for "novel" = 18 months
TRACK_H=52           # track an entity for up to 1 year after it emerges
TERM_STOP = set(ENGLISH_STOP_WORDS) | {"says","said","new","year","week","report","shares","stock",
    "jan","feb","mar","apr","may","jun","jul","aug","sep","oct","nov","dec"}
EVENT_STOP = {"collapse","rout","bankruptcy","bankrupt","fraud","lawsuit","sue","sues","sued","probe","hearing",
 "trial","court","arrest","arrested","resign","resigns","ban","bans","banned","outage","recall","default","slump",
 "slumps","plunge","plunges","crash","derailment","strike","quake","earthquake","protest","protests","unrest",
 "attack","war","sanctions","fine","fined","scandal","layoffs","layoff","pivot","halt","halts","delay","delays",
 "death","dies","killed","guilty","charges","charged","indicted","crisis","takeover","merger","deal","acquires",
 "acquire","buys","stake","ipo","listing","bond","bonds","notes","debt","offering","case","settlement","shortage",
 "shutdown","tumbles","soars","jumps","rises","falls","drops","gains","cuts","raises"}
def is_entity(t): return not any(tok in EVENT_STOP for tok in t.split())
HINT = re.compile(r"\bionq\b|rigetti|\bd-wave\b|\bdwave\b|quantinuum|\bqubit|quantum comput", re.I)
def normalize_terms(v): return v.tolist() if isinstance(v, np.ndarray) else (list(v) if isinstance(v,(list,tuple)) else [])
def filter_terms(ts): return [t for t in ts if len(t)>=3 and not any(tok in TERM_STOP for tok in t.split())]
def week_ts(w): return pd.Period(w, freq=FREQ).start_time
def clustcoef(P, adj):
    if len(P)<2: return 0.0
    tot=pairs=0
    for i in range(len(P)):
        for j in range(i+1,len(P)):
            tot+=1
            if P[j] in adj[P[i]]: pairs+=1
    return pairs/tot if tot else 0.0
print(f"setup ok · rolling baseline {ROLL_WEEKS}wk (~18mo) · track horizon {TRACK_H}wk")

setup ok · rolling baseline 78wk (~18mo) · track horizon 52wk


In [2]:
def detect_rolling(news, ds, de, roll_weeks, track_h):
    """Point-in-time novelty: an entity EMERGES the week it is not seen in the trailing
    `roll_weeks` weeks, then is tracked forward for `track_h` weeks (so growth is visible)."""
    df = news.copy()
    df["terms_f"] = df["terms"].map(normalize_terms).map(filter_terms)
    df["week"] = df["date"].dt.to_period(FREQ).astype(str)
    all_weeks = sorted(df["week"].unique(), key=week_ts)
    last_seen, emerged_at, rows = {}, {}, []
    for i, week in enumerate(all_weeks):
        grp = df.loc[df.week == week]
        twc = Counter()
        for tl in grp["terms_f"]:
            for t in set(tl): twc[t] += 1
        if ds <= week_ts(week) <= de:                                  # only discovery weeks emit
            adj = defaultdict(Counter)
            for tl in grp["terms_f"]:
                for a, b in combinations(sorted(set(tl)), 2): adj[a][b]+=1; adj[b][a]+=1
            for t, cnt in twc.items():
                if cnt < MIN_MENTIONS or not is_entity(t): continue
                novel = (t not in last_seen) or (last_seen[t] < i - roll_weeks)   # rolling novelty
                if novel and t not in emerged_at: emerged_at[t] = i              # record emergence
                if t in emerged_at and (i - emerged_at[t]) <= track_h:           # track forward
                    epart = [p for p,_ in adj[t].most_common() if is_entity(p)]
                    P = epart[:CLUSTER_K]
                    reps = [hl for hl,tl in zip(grp["Headline"],grp["terms_f"]) if t in set(tl)][:REP_HL]
                    rows.append({"week":week,"anchor":t,"mentions":int(cnt),"anchor_degree":len(epart),
                                 "clustering":round(clustcoef(P,adj),3),"subgraph":[t]+P[:10],"rep_headlines":reps})
        for t in twc: last_seen[t] = i                                 # update AFTER (point-in-time)
    return pd.DataFrame(rows)

In [3]:
# 2019-2020 baseline history + 2021-2023 discovery (history needed for the trailing window)
base = pd.read_parquet(OUTPUT_DIR/"baseline_2019_2020_terms.parquet")
disc = pd.read_parquet(OUTPUT_DIR/"genai_graph_terms.parquet")[["Headline","date","terms"]]
base["date"]=pd.to_datetime(base["date"]).dt.normalize(); disc["date"]=pd.to_datetime(disc["date"]).dt.normalize()
news = pd.concat([base, disc], ignore_index=True); news = news[news.date<=DISCOVERY_END]
print(f"corpus: {len(news):,} headlines")
R = detect_rolling(news, DISCOVERY_START, DISCOVERY_END, ROLL_WEEKS, TRACK_H)
R["q"] = R["anchor"].str.contains(HINT, na=False)
print(f"{len(R):,} (entity x week) rows · {R.anchor.nunique():,} distinct emerged entities  "
      f"(was 47,042 with the fixed baseline)")
print("\nquantum emerged anchors:")
print(R.loc[R.q,["week","anchor","mentions","anchor_degree","clustering"]].sort_values(["week","anchor"]).head(20).to_string(index=False))

corpus: 6,974,403 headlines
53,802 (entity x week) rows · 44,877 distinct emerged entities  (was 47,042 with the fixed baseline)

quantum emerged anchors:
                 week anchor  mentions  anchor_degree  clustering
2021-02-23/2021-03-01   ionq         4              5       1.000
2021-03-02/2021-03-08   ionq         4             12       0.439
2021-09-21/2021-09-27   ionq         3             14       0.538
2021-09-28/2021-10-04   ionq         5             13       0.692
2021-11-16/2021-11-22   ionq         4             19       0.314


In [9]:
R.tail()

,week,anchor,mentions,anchor_degree,clustering,subgraph,rep_headlines,q,caught
53797,2023-12-26/2024-01-01,short sbb,3,3,1.000,"[short sbb, sbb, short, viceroy]","[*VICEROY SAYS IT IS SHORT SBB, Viceroy Says I...",False,False
53798,2023-12-26/2024-01-01,ataturk,3,23,0.486,"[ataturk, saudi, turkish, ataturk t-shirts, fe...","[*GALATASARAY, FENERBAHCE REACT TO SAUDI BAN O...",False,True
53799,2023-12-26/2024-01-01,floating ipab,3,4,1.000,"[floating ipab, floating, ipab, mexico, mxn1]",[Mexico to Sell MXN1.5B 2030 Floating IPAB Bon...,False,False
53800,2023-12-26/2024-01-01,buzzfeed close,3,7,0.952,"[buzzfeed close, buzzfeed, close, complex, inf...",[*BUZZFEED CLOSE TO SELLING COMPLEX FOR ABOUT ...,False,False
53801,2023-12-26/2024-01-01,honors list,3,11,0.891,"[honors list, aviva, honors, james, joins, lis...",[James Bond Singer Bassey Joins Aviva Chief on...,False,True


In [4]:
R["caught"]=(R.mentions>=MIN_MENTIONS)&(R.anchor_degree>=DEGREE_MIN)
def promote(R):
    out=[]
    for a,sub in R[R.caught].groupby("anchor"):
        sub=sub.sort_values("week", key=lambda s:s.map(week_ts))
        wk,deg,clu=list(sub.week),list(sub.anchor_degree),list(sub.clustering)
        p=None
        for i in range(len(wk)):
            if (i+1)>=PERSIST_WEEKS and deg[i]>=max(deg[:i] or [0]) and float(np.median(clu[:i+1]))<=CLUSTER_MAX:
                p=wk[i]; break
        out.append({"anchor":a,"first_caught":wk[0],"n_weeks":len(wk),"deg_max":max(deg),
                    "med_clustering":round(float(np.median(clu)),3),"promoted_week":p,"q":bool(HINT.search(a))})
    return pd.DataFrame(out)
P=promote(R); promoted=P[P.promoted_week.notna()].copy()
print(f"caught {R[R.caught].anchor.nunique():,}  ·  promoted {len(promoted)}  (was 37,244 / 1,312)")
print("\nquantum lifecycle:")
print(P[P.q].sort_values("first_caught")[["anchor","first_caught","n_weeks","deg_max","med_clustering","promoted_week"]].head(12).to_string(index=False))

caught 34,941  ·  promoted 626  (was 37,244 / 1,312)

quantum lifecycle:
anchor          first_caught  n_weeks  deg_max  med_clustering         promoted_week
  ionq 2021-03-02/2021-03-08        4       19           0.489 2021-09-21/2021-09-27


In [5]:
from typing import Literal
from pydantic import BaseModel
pset=set(promoted.anchor)
G=nx.Graph(); G.add_nodes_from(pset)
for _,r in R[R.anchor.isin(pset)&R.caught].iterrows():
    for p in r.subgraph:
        if p in pset and p!=r.anchor: G.add_edge(r.anchor,p)
groups=[sorted(c) for c in nx.connected_components(G)]
sizes=sorted([len(g) for g in groups], reverse=True)
print(f"themes: {len(groups)} · biggest component {sizes[0]} (was 835)")
def members(anchors):
    s=set()
    for sg in R[R.anchor.isin(anchors)&R.caught].subgraph: s.update(sg)
    return sorted(s)
def evidence(anchors):
    seen,out=set(),[]
    for h in R[R.anchor.isin(anchors)&R.caught].sort_values("week",key=lambda s:s.map(week_ts)).rep_headlines:
        for hl in h:
            if hl.lower() not in seen: seen.add(hl.lower()); out.append(hl)
    return out[::max(1,len(out)//10)][:10] if len(out)>10 else out
gdf=pd.DataFrame({"anchors":groups})
gdf["reach"]=gdf.anchors.map(lambda a:int(P.set_index("anchor").loc[a,"deg_max"].max()))
gdf["q"]=gdf.anchors.map(lambda a:any(HINT.search(x) for x in a))
gdf["subgraph"]=gdf.anchors.map(members)
gdf=gdf.sort_values("reach",ascending=False).reset_index(drop=True)
REJECT_SYS=("You are a conservative FILTER that REMOVES clusters of news headlines that are NOT emerging themes. "
 "You never decide what IS a theme; only flag clusters that CLEARLY are: 1. single_entity_event, "
 "2. macro_market_aggregate, 3. boilerplate_wire. If unclear, KEEP. KEEP anything describing a SPECIFIC "
 "technological/industrial/product development across multiple actors.")
class Reject(BaseModel):
    verdict: Literal["keep","reject"]; category: Literal["single_entity_event","macro_market_aggregate","boilerplate_wire","none"]; reason: str
_client=None
def judge(sg,hl):
    global _client
    from openai import OpenAI
    if _client is None: _client=OpenAI(api_key=os.environ["OPENAI_API_KEY"], base_url=os.environ.get("OPENAI_BASE") or None)
    user="Cluster entities: "+", ".join(sg[:14])+"\nHeadlines:\n"+"\n".join(f"- {h}" for h in hl)+"\n\nClassify this cluster."
    r=_client.beta.chat.completions.parse(model=os.environ.get("OPENAI_DEFAULT_MODEL","gpt-4o-mini"),temperature=0,
        response_format=Reject, messages=[{"role":"system","content":REJECT_SYS},{"role":"user","content":user}])
    p=r.choices[0].message.parsed; return (p.verdict=="keep"),p.category
gdf["llm_keep"],gdf["llm_cat"]=None,None
for i,g in gdf.head(LLM_MAX_GROUPS).iterrows():
    k,c=judge(g.subgraph, evidence(g.anchors)); gdf.at[i,"llm_keep"],gdf.at[i,"llm_cat"]=k,c
print(f"LLM kept {int((gdf.llm_keep==True).sum())}")

themes: 550 · biggest component 15 (was 835)
LLM kept 3


In [6]:
qc=R[R.q&R.caught]; qfirst=qc.week.min() if len(qc) else None
gp=P[P.q&P.promoted_week.notna()]; qprom=gp.promoted_week.min() if len(gp) else None
print("="*66)
print("QUANTUM (rolling baseline, point-in-time):")
print(f"  first CAUGHT : {qfirst}")
print(f"  PROMOTED     : {qprom}")
if qprom:
    lead=(WQTM-week_ts(qprom)).days; print(f"  lead vs WQTM : {lead} days (~{lead//365}y {(lead%365)//30}m)")
qg=gdf[gdf.q]
if len(qg): print(f"  quantum theme : {', '.join(qg.iloc[0].subgraph[:14])}\n  LLM verdict   : {'KEEP' if qg.iloc[0].llm_keep else 'reject/'+str(qg.iloc[0].llm_cat)}")
print("="*66)
print(f"funnel: {R.anchor.nunique():,} emerged -> {R[R.caught].anchor.nunique():,} caught -> {len(promoted)} promoted -> {len(groups)} themes -> {int((gdf.llm_keep==True).sum())} LLM-kept")
print("\nLLM-kept themes (shortlist):")
for _,g in gdf[gdf.llm_keep==True].head(15).iterrows():
    print(f"  reach {int(g.reach):3}{' Q' if g.q else '  '}  {', '.join(g.subgraph[:8])}")
R.to_parquet(OUTPUT_DIR/"quantum_rolling_anchor_weeks.parquet", index=False)
P.to_parquet(OUTPUT_DIR/"quantum_rolling_promotion.parquet", index=False)
print("\nsaved -> quantum_rolling_*.parquet")

QUANTUM (rolling baseline, point-in-time):
  first CAUGHT : 2021-03-02/2021-03-08
  PROMOTED     : 2021-09-21/2021-09-27
  lead vs WQTM : 1479 days (~4y 0m)
  quantum theme : announce, announce closing, approve, begin, begin trading, boosts, cadence, cadence design, capital, capitalization, cirrus, cirrus logic, closing, design
  LLM verdict   : reject/None
funnel: 44,877 emerged -> 34,941 caught -> 626 promoted -> 550 themes -> 3 LLM-kept

LLM-kept themes (shortlist):
  reach 134    affecting, all-time, all-time high, amex, amid, appetite, asia, bernstein
  reach 119    access, act, act set, add, add bing, adds, administration, advice
  reach  80    accounting, ankara, arabia, attract, attract capital, aybala, aybala simsek, bank

saved -> quantum_rolling_*.parquet
